# 15 — Lab trial pack: what the laboratory actually prepares and tests

**What this is.** One Excel workbook, `15_lab_trial_pack.xlsx`, built from the outputs of
notebooks 12 (the M30 / 120 mm / severe order) and 14 (every grade in the dataset). For each
order it gives the 5 designs, the **3 IS 10262 cl 5.8 trial batches**, a **batch sheet the
lab can weigh from** (with the day's aggregate moisture typed in and every weight
recalculating), the tests to run with their IS clause, the acceptance checks, a results log,
and the IS 10262 cl 5.8.1 report checklist.

**Why it matters here.** A computed mix is not a mix. IS 10262 cl 5.8: *"The calculated mix
proportions shall be checked by means of trial batches."* This pack is the hand-over from
the SDL to the laboratory.

**What it never does.** It claims no result. Every mix in it is `COMPUTATIONALLY_COMPLIANT`
and nothing more. The laboratory, then the plant, move it up the ladder (notebook 13).

## How a lab uses it

1. **Before batching** — sheet *Material checks*: confirm the certificates (cement, fly ash,
   admixture datasheet) and measure aggregate absorption and moisture (IS 2386 Part 3).
2. **Batch** — sheet *Batch sheets*: type the measured moisture into the yellow cells; the
   weights and the corrected water update (IS 10262 Annex A-10 NOTE, A-11). For the fine
   aggregate you may type either the total moisture (% of dry mass, the plant's oven-dry or
   meter reading) or the surface moisture by IS 2386 (Part 3) cl 5.5.1 (% of SSD mass) —
   the sheet converts the second into the first.
3. **Trial Mix 1** — measure slump. If it is off the stipulated value, adjust admixture (or
   water) keeping the w/cm, and recompute: that is Trial Mix 2 (IS 10262 cl 5.8, A-14).
4. **Trials 3 and 4** — same water **as Trial 2**, w/cm about −10 % and +10 % (cl 5.8). The
   pre-built sheets assume Trial 1 needed no water correction; if it did, rebuild them with
   `layer0.trial_set(..., trial2_water_kg_m3=...)`. Cast 3 cubes for 7 days and 3 for
   28 days from each — `[PC]` the ages IS 516 (Part 1/Sec 1) cl 3.3 calls "most usual";
   cl 3.4 asks for at least three per age.
5. **Record** — sheet *Results log*. Plot 28-day strength against w/cm and read off the
   w/cm for the target mean strength (Annex A-15). Report per cl 5.8.1.

Run order: `09 → 10 → 10b → 11 → 12 → 14 → 15` (13 optional).

In [ ]:
import sys as _sys
for _m in ("is_rules", "layer0", "workability_residual", "strength_residual",
           "screen", "selection", "loop"):
    _sys.modules.pop(_m, None)

import json
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

import is_rules as IS

pd.set_option("display.width", 200)

## 1. Collect the orders from notebooks 12 and 14

In [ ]:
def _read(path):
    p = Path(path)
    if not p.exists():
        print(f"missing {path} -- run the notebook that writes it first")
        return pd.DataFrame()
    return pd.read_csv(p)


meta12 = json.loads(Path("12_select_meta.json").read_text()) if Path("12_select_meta.json").exists() else {}
D12, T12, B12, E12 = (_read(f) for f in ("12_next_designs.csv", "12_next_experiments.csv",
                                          "12_trial_batches.csv", "12_explore_proposals.csv"))
S14, D14, T14, B14, E14 = (_read(f) for f in ("14_all_grades_summary.csv", "14_all_grades_designs.csv",
                                               "14_all_grades_trials.csv", "14_all_grades_trial_batches.csv",
                                               "14_all_grades_explore_proposals.csv"))

DESIGNS = pd.concat([D12, D14], ignore_index=True)
TRIALS = pd.concat([T12, T14], ignore_index=True)
BATCHES = pd.concat([B12, B14], ignore_index=True)
EXPLORE = pd.concat([E12, E14], ignore_index=True)

orders = []
if len(T12) or len(D12):
    orders.append({"order_id": "M30-120-severe", "grade": "M30", "exposure": "SEVERE", "use": "RCC",
                   "target_slump_mm": 120.0, "status": meta12.get("status", "?"),
                   "source": "notebook 12 -- the M30 order, plant Plant D materials",
                   "notes": " | ".join(meta12.get("notes", []))})
for _, r in S14.iterrows():
    orders.append({"order_id": r["family"], "grade": r["grade"], "exposure": r["exposure"],
                   "use": r["use"], "target_slump_mm": r["target_slump_mm"], "status": r["status"],
                   "source": "notebook 14 -- the family's recorded order",
                   "notes": r.get("reason") if isinstance(r.get("reason"), str) else r.get("notes", "")})
ORDERS = pd.DataFrame(orders)
print(ORDERS[["order_id", "grade", "exposure", "target_slump_mm", "status"]].to_string(index=False))
print(f"\n{len(DESIGNS)} designs, {len(TRIALS)} trial batches, {len(EXPLORE)} exploration proposals")

## 2. Build the workbook

Yellow cells are inputs the lab fills on the day. Everything else is either a design value
from the pipeline or an Excel formula. The formulas are the IS 10262 Annex A-11 mass balance:
weighed mass = SSD mass ÷ (1 + absorption) × (1 + moisture); water to add = design water +
Σ (SSD mass − weighed mass).

In [ ]:
YELLOW = PatternFill("solid", fgColor="FFF2CC")
GREY = PatternFill("solid", fgColor="E7E6E6")
BLUE = PatternFill("solid", fgColor="DDEBF7")
BOLD = Font(bold=True)
TITLE = Font(bold=True, size=13)
WRAP = Alignment(wrap_text=True, vertical="top")
THIN = Border(*(Side(style="thin", color="999999"),) * 4)

wb = Workbook()


def sheet(title, widths):
    ws = wb.create_sheet(title)
    for i, w in enumerate(widths, start=1):
        ws.column_dimensions[get_column_letter(i)].width = w
    return ws


def put(ws, row, values, *, bold=False, fill=None, wrap=False, fmt=None):
    for j, v in enumerate(values, start=1):
        c = ws.cell(row=row, column=j, value=(None if (isinstance(v, float) and np.isnan(v)) else v))
        if bold:
            c.font = BOLD
        if fill:
            c.fill = fill
        if wrap:
            c.alignment = WRAP
        if fmt and isinstance(v, (int, float)):
            c.number_format = fmt
    return row + 1


def df_block(ws, row, df, cols, heads=None, fmt="0.000"):
    row = put(ws, row, heads or cols, bold=True, fill=GREY, wrap=True)
    for _, r in df.iterrows():
        row = put(ws, row, [r.get(c) for c in cols], fmt=fmt)
    return row + 1


# ------------------------------------------------------------------ READ ME
ws = wb.active
ws.title = "READ ME"
ws.column_dimensions["A"].width = 30
ws.column_dimensions["B"].width = 110
r = put(ws, 1, ["RMC SDL -- laboratory trial pack"], bold=True)
ws["A1"].font = TITLE
lines = [
    ("Validation level", "Every mix in this workbook is COMPUTATIONALLY_COMPLIANT only. None has been cast. "
                         "It becomes LAB_VALIDATED only after the trials below are cast and tested."),
    ("What to cast", "Sheet 'Trials': 3 batches per order -- IS 10262 cl 5.8 Trial Mix 1 (-> 2 after any slump "
                     "correction), Trial Mix 3 (same water, w/cm -10 %), Trial Mix 4 (same water, w/cm +10 %). [EI] If "
                     "+10 % would break IS 456 Table 5, Trial 4 is held at the limit (after Annex A-15: 'durability "
                     "requirement shall be met'); if that leaves it within 2 % of Trial 1 it is replaced by w/cm -5 % "
                     "so the A-15 graph has three distinct points. Trials 3 and 4 use the water of Trial 2: the sheets "
                     "assume Trial 1 needed no water correction -- if it did, rebuild them."),
    ("Designs", "Sheet 'Designs': up to 5 per order, ranked on cost and carbon (PLACEHOLDER rates), every one "
                "inside the data the models were fitted on and through the full IS gate."),
    ("Exploration", "Sheet 'Exploration': optional proposals chosen where the models know least, inside a "
                    "trust region around the data. Expert review before casting."),
    ("Declared inputs", "Fly ash specific gravity 2.20 -- ASSUMED (IS 10262 Annex B-2(d) worked-example value; "
                        "not in the dataset; replace with the fly ash certificate). Admixture dosage range "
                        "0.621-1.476 % of binder -- PLANT-OBSERVED, not the manufacturer's declaration required by "
                        "IS 9103 cl 10.2(f); replace with the PCE datasheet range."),
    ("Moisture", "The batch sheets open with the plant's MEDIAN aggregate moisture from the dataset. Measure "
                 "it on the day and type it into the yellow cells (absorption: IS 2386 Part 3 cl 2; surface "
                 "moisture of fine aggregate: cl 5)."),
    ("IS 456 cl 5.1, 5.2.1", "[EI] This plant uses PPC with added fly ash, read as an 'other combination' under "
                      "IS 456 cl 5.1: allowed only with satisfactory suitability data; the plant's own 256 batches with "
                      "three 28-day cubes each are recorded as that evidence [EI]. Fly ash + GGBS together is NOT "
                      "allowed without its own suitability data. Fly ash is a pozzolana: cl 5.2.1 requires the "
                      "permission of the engineer-in-charge -- sign it off on 'Material checks'."),
    ("Standards used", "IS 10262:2019 (cl 4.2, 5.1, 5.1.1 NOTE, 5.3-5.8, 5.8.1, Annex A-10 to A-15); IS 456:2000 "
                       "(Table 5 + Note 1, cl 5.1, 5.2.1.1, 5.2.2, 5.4, 8.2.4.2); IS 455:2015 cl 5; "
                       "IS 1489 (Part 1):1991 cl 5 (Amd 3); IS 4926:2003 cl 5.2.2.1, 6.2.1; IS 1199 (Parts 1, 2, "
                       "3, 5):2018; IS 516 (Part 1/Sec 1):2021 cl 3.3-3.6; IS 2386 (Part 3):1963; IS 9103:1999 cl 10.2."),
    ("What this is not", "It is a decision-support and experimental-design output. It does not replace "
                         "engineering judgement, laboratory validation, plant quality control, or contractual "
                         "requirements."),
]
r += 1
for k, v in lines:
    ws.cell(row=r, column=1, value=k).font = BOLD
    c = ws.cell(row=r, column=2, value=v)
    c.alignment = WRAP
    ws.row_dimensions[r].height = 15 * max(1, len(v) // 100 + 1)
    r += 1

# ------------------------------------------------------------------ Orders
ws = sheet("Orders", [20, 7, 11, 6, 10, 34, 46, 90])
r = put(ws, 1, ["Orders in this pack"], bold=True)
df_block(ws, 3, ORDERS, ["order_id", "grade", "exposure", "use", "target_slump_mm", "status", "source", "notes"],
         ["order", "grade", "exposure", "use", "target slump mm", "status", "source", "notes"], fmt="0")

# ------------------------------------------------------------------ Designs
ws = sheet("Designs", [18, 7, 9, 9, 9, 9, 9, 8, 8, 8, 9, 8, 9, 8, 8, 10, 9])
cols = ["order_id", "design_rank", "binder_kg_m3", "cement_kg_m3", "flyash_kg_m3", "water_kg_m3",
        "admixture_kg_m3", "sp_pct_binder", "wb_ratio", "wc_counted", "fc_pred", "fc_sd", "target_mean",
        "P_strength", "P_slump", "cost_inr_m3", "on_pareto_front"]
heads = ["order", "rank", "binder kg/m3", "cement kg/m3", "fly ash kg/m3", "free water kg/m3",
         "admixture kg/m3", "SP % binder", "w/cm (total)", "w/c counted (IS 456 T5 N1, incl. admix water)",
         "predicted 28 d (sample, MPa)", "sd MPa", "target mean MPa (IS 10262 cl 4.2)",
         "P(strength)", "P(slump in IS 4926 band)", "cost Rs/m3 (PLACEHOLDER)", "on cost-carbon front"]
put(ws, 1, ["Designs -- COMPUTATIONALLY_COMPLIANT. Fine and coarse aggregate in the 'Trials' and 'Batch sheets' tabs."], bold=True)
df_block(ws, 3, DESIGNS, cols, heads)

# ------------------------------------------------------------------ Trials
ws = sheet("Trials", [18, 6, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 9, 8, 9, 9, 8, 70])
cols = ["order_id", "trial_no", "binder_kg_m3", "cement_kg_m3", "flyash_kg_m3", "water_kg_m3",
        "admixture_kg_m3", "fine_agg_kg_m3", "coarse_agg_kg_m3", "sp_pct_binder", "wb_ratio", "wc_counted",
        "fc_pred", "fc_sd", "target_mean", "slump_pred", "slump_sd", "trial_note"]
heads = ["order", "trial mix no.", "binder kg/m3", "cement kg/m3", "fly ash kg/m3", "free water kg/m3",
         "admixture kg/m3", "fine agg SSD kg/m3", "coarse agg SSD kg/m3", "SP % binder", "w/cm (total)",
         "w/c counted", "predicted 28 d (sample, MPa)", "sd MPa", "target mean MPa", "predicted slump mm",
         "slump sd mm", "what this trial is (IS 10262 cl 5.8)"]
put(ws, 1, ["Trial batches per IS 10262 cl 5.8 -- same water in all three; only the binder moves"], bold=True)
df_block(ws, 3, TRIALS, cols, heads)

In [ ]:
# ------------------------------------------------------------------ Batch sheets (formulas)
ws = sheet("Batch sheets", [34, 16, 16, 16, 60])
r = put(ws, 1, ["Batch sheets -- type the day's moisture into the yellow cells; weights recalculate "
                "(IS 10262 Annex A-10 NOTE, A-11)"], bold=True)
r += 1
CHECK = []          # (cell of the weighed water, python value) for the verification step
for (oid, tno), g in BATCHES.groupby(["order_id", "trial_no"], sort=False):
    g = g.reset_index(drop=True)
    V = float(g["batch_m3"].iloc[0])
    r = put(ws, r, [f"{oid} -- Trial Mix {int(tno)}"], bold=True, fill=BLUE)
    rv = r;  r = put(ws, r, ["batch volume, m3", V], fmt="0.000")
    ra_s = r; r = put(ws, r, ["fine aggregate absorption, % (IS 2386 Pt 3 cl 2)", g["sand_absorption_pct"].iloc[0]], fmt="0.00")
    ra_c = r; r = put(ws, r, ["coarse aggregate absorption, % (IS 2386 Pt 3 cl 2)", g["ca_absorption_pct"].iloc[0]], fmt="0.00")
    rt_s = r; r = put(ws, r, ["fine aggregate TOTAL moisture today, % of dry mass (oven-dry / meter)",
                              g["sand_moisture_pct_used"].iloc[0]], fmt="0.00")
    ws.cell(row=rt_s, column=2).fill = YELLOW
    rp_s = r; r = put(ws, r, ["OR fine aggregate SURFACE moisture, % of SSD (IS 2386 Pt 3 cl 5.5.1) -- if "
                              "filled, it overrides the row above"], fmt="0.00")
    ws.cell(row=rp_s, column=2).fill = YELLOW
    rm_s = r
    ws.cell(row=r, column=1, value="fine aggregate moisture used, % of dry mass  [= a + Ps x (1 + a/100) if Ps given]")
    ws.cell(row=r, column=2, value=f"=IF(ISBLANK(B{rp_s}),B{rt_s},B{ra_s}+B{rp_s}*(1+B{ra_s}/100))").number_format = "0.00"
    r += 1
    rm_c = r; r = put(ws, r, ["coarse aggregate TOTAL moisture today, % of dry mass (oven-dry / meter)",
                              g["ca_moisture_pct_used"].iloc[0]], fmt="0.00")
    ws.cell(row=rm_c, column=2).fill = YELLOW
    r = put(ws, r, ["material", "design SSD kg/m3", "SSD kg in batch", "WEIGH kg", "basis"],
            bold=True, fill=GREY)
    agg_rows, water_row, first = [], None, r
    for _, x in g.iterrows():
        m = x["material"]
        ws.cell(row=r, column=1, value=m)
        ws.cell(row=r, column=2, value=float(x["design_ssd_kg_m3"])).number_format = "0.00"
        ws.cell(row=r, column=3, value=f"=B{r}*$B${rv}").number_format = "0.000"
        if m.startswith("fine"):
            f = f"=C{r}/(1+$B${ra_s}/100)*(1+$B${rm_s}/100)"
            basis = "A-11: SSD / (1 + absorption) x (1 + moisture)"
            agg_rows.append(r)
        elif m.startswith("coarse"):
            f = f"=C{r}/(1+$B${ra_c}/100)*(1+$B${rm_c}/100)"
            basis = "A-11: SSD / (1 + absorption) x (1 + moisture); split per A-13 (plant 20:10 ratio)"
            agg_rows.append(r)
        elif m.startswith("water"):
            f, basis, water_row = None, "", r
        elif m.startswith("admixture"):
            f, basis = f"=C{r}", "listed separately from the mixing water (Annex A-10); [EI] its water counts in the w/c check only (cl 5.1.1 NOTE), not subtracted here"
        else:
            f, basis = f"=C{r}", "as designed"
        if f:
            ws.cell(row=r, column=4, value=f).number_format = "0.000"
        ws.cell(row=r, column=5, value=basis)
        r += 1
    adj = "+".join(f"(C{a}-D{a})" for a in agg_rows)
    ws.cell(row=water_row, column=4, value=f"=C{water_row}+{adj}").number_format = "0.000"
    ws.cell(row=water_row, column=5, value="A-11: design water + SSD - weighed, for every aggregate "
                                           "(negative correction = water the wet aggregate brings in)")
    py_w = float(g.loc[g["material"].str.startswith("water"), "batch_kg"].iloc[0])
    CHECK.append((f"D{water_row}", py_w, oid, int(tno)))
    r += 1

# ------------------------------------------------------------------ Tests & acceptance
ws = sheet("Tests and acceptance", [36, 42, 70])
r = put(ws, 1, ["Tests for every trial batch, with the clause that sets each one"], bold=True)
r = put(ws, 3, ["test", "standard / clause", "what to do / accept"], bold=True, fill=GREY)
tests = [
    ("Sampling", "IS 1199 (Part 1):2018 cl 6, 6.3", "Composite sample; strength sample at least 0.02 m3 (cl 6). Start "
     "slump, temperature and air within 5 min (recommended); start filling cubes within 15 min (cl 6.3)."),
    ("Concrete temperature", "IS 1199 (Part 1):2018 cl 6.4", "Record at sampling (plant median in the data: about 30 C)."),
    ("Slump at mixing", "IS 1199 (Part 2):2018", "Mould 200/100/300 mm (cl 4.3.2.1). True slump only; valid "
     "10-210 mm (cl 4.3.1, 4.3.4)."),
    ("Slump after transit time", "IS 4926:2003 cl 5.2.2.1 [EI]", "As a general requirement RMC is discharged within 2 h "
     "of loading; a longer period may be permitted with retarding admixtures, in cool humid weather or for "
     "chilled concrete. The plant records slump at 45 min -- repeat at that time to check retention."),
    ("Slump band for the trial", "IS 4926:2003 cl 6.2.1 [EI]", "IS 10262 cl 5.8 gives no tolerance; the RMC delivery "
     "band is used: specified value +/- 25 mm or +/- 1/3 of it, whichever is less. Outside it: adjust admixture or "
     "water keeping the pre-selected w/cm -> Trial Mix 2 (IS 10262 cl 5.8; rework and check durability, A-14)."),
    ("Segregation, bleeding, finish", "IS 10262 cl 5.8", "Observe and record for Trial Mix 1 ('freedom from segregation "
     "and bleeding and its finishing properties')."),
    ("Fresh density and volume per batch", "IS 1199 (Part 3):2018 cl 4.2.1 + Table 1, 4.6.2", "Container per Table 1: "
     "250 mm dia x 280 mm high for aggregate up to 40 mm (about 13.7 L). Volume per batch Vb = total batch mass / "
     "density (cl 4.6.2); compare with the batch volume."),
    ("Cubes", "IS 1199 (Part 5):2018 cl 4.1.1", "150 mm cubes. [PC] 3 for 7 days + 3 for 28 days from each trial "
     "(the ages IS 516 Pt 1/Sec 1 cl 3.3 calls most usual; cl 3.4: at least three per age)."),
    ("Curing", "IS 1199 (Part 5):2018 cl 7", "In the mould 16 h to 3 days at 27 +/- 3 C, protected; then in water "
     "at 27 +/- 2 C (or a chamber at 27 +/- 2 C, RH >= 95 %) until just before testing."),
    ("Compression test", "IS 516 (Part 1/Sec 1):2021 cl 3.5.2", "Load at 14 N/mm2 per min, no shock."),
    ("Cube result for one age", "IS 516 (Part 1/Sec 1):2021 cl 3.6", "Average of 3 if every cube is within +/- 15 % "
     "of the average; otherwise repeat (or, with no further sample, mean of the two closest). Report to 0.5 MPa."),
    ("Spread rule conflict", "IS 456 cl 15.4 vs IS 516 cl 3.6", "For an ACCEPTANCE sample IS 456 cl 15.4 says a spread "
     "over +/- 15 % makes the result INVALID; IS 516 says repeat. [PC] Trial cubes follow IS 516; acceptance "
     "samples follow IS 456."),
    ("Judge a trial", "IS 10262 cl 4.2; IS 456 cl 15.3 [EI]", "Compare the 28-day result (mean of 3 = one sample) with "
     "the target mean strength. 7-day results steer only: IS 456 cl 15.1.1 makes 28 days the acceptance criterion."),
    ("Choose the final w/cm", "IS 10262 cl 5.1, 5.8, Annex A-15", "Plot 28-day strength against w/cm for Trials 2-4 "
     "and read the w/cm for the target mean strength. The durability requirement (IS 456 Table 5) must still be met."),
    ("Field trial", "IS 10262 cl 5.8", "Additional field trials are recommended, produced by the plant's actual "
     "production method."),
]
for t in tests:
    r = put(ws, r, list(t), wrap=True)

# ------------------------------------------------------------------ Material checks
ws = sheet("Material checks", [34, 40, 70])
r = put(ws, 1, ["Before batching -- confirm every material (fill the yellow column)"], bold=True)
r = put(ws, 3, ["material", "standard / clause", "what to confirm"], bold=True, fill=GREY)
mats = [
    ("Cement (PPC)", "IS 1489 (Part 1):1991 + Amd 1-5", "Manufacturer's certificate: declared fly ash % (15-35, Amd 3), "
     "soundness, setting, 3/7/28-day strength. The declared % lets more added fly ash count (IS 456 Table 5 Note 1)."),
    ("Cement (OPC), if used", "IS 269:2015 Tables 2, 3; cl 11.1", "Grade-wise limits; cement more than 3 months old to be "
     "tested before use."),
    ("Fly ash", "IS 3812 (Part 1):2013; IS 456 cl 5.2.1.1", "Supplier certificate. Its SPECIFIC GRAVITY replaces the assumed "
     "2.20 used here."),
    ("Admixture (PCE)", "IS 9103:1999 cl 10.2", "Datasheet: recommended dosage range (replaces the plant-observed "
     "0.621-1.476 %), solids content, chloride."),
    ("Fine aggregate (M-sand)", "IS 383:2016 Table 9, cl 6.3; IS 2386 (Part 3)", "Grading zone from the 600 um sieve; "
     "specific gravity and absorption (Pt 3 cl 2, % of dry mass); moisture on the day (surface moisture by "
     "cl 5.5.1 is % of SSD mass -- the batch sheet converts it)."),
    ("Coarse aggregate", "IS 383:2016; IS 10262 Annex A-13; IS 2386 (Part 3)", "20-10 and 10-4.75 mm split to meet IS 383 "
     "Table 7 [Table 7 not encoded -- check in the lab]; specific gravity, absorption, moisture on the day."),
    ("Water", "IS 456:2000 cl 5.4", "Clean, free from injurious amounts of oils, acids, alkalis, salts, sugar, organic "
     "materials; potable water is generally satisfactory. pH not less than 6; solids within the cl 5.4 table limits; "
     "in case of doubt the cl 5.4.1 tests (90 % cube strength, setting time)."),
    ("Engineer-in-charge permission", "IS 456:2000 cl 5.2.1", "Fly ash is a pozzolana: used 'with the permission of the "
     "engineer-in-charge'. Record the name and date."),
    ("Combination", "IS 456:2000 cl 5.1, Table 5 Note 1", "[EI] PPC + added fly ash is read as an 'other combination': "
     "suitability data recorded (the plant's 256 batches with 28-day cubes). Note 1 counts additions in cement "
     "content and w/c only 'if the suitability is established'. Do not add GGBS without its own suitability data."),
]
for m_ in mats:
    r = put(ws, r, list(m_), wrap=True)

# ------------------------------------------------------------------ Results log
ws = sheet("Results log", [18, 8, 10, 11, 11, 10, 9, 8, 8, 8, 9, 8, 8, 8, 10, 12, 30])
r = put(ws, 1, ["Results log -- yellow = measured. A human rejection without casting is NOT a result: write "
                "the reason in the last column and leave the measurements blank."], bold=True)
heads = ["order", "trial no.", "date cast", "slump 0 min mm", "slump 45 min mm", "temp C", "density kg/m3",
         "7d cube 1", "7d cube 2", "7d cube 3", "7d result", "28d cube 1", "28d cube 2", "28d cube 3",
         "28d result", "every 28d cube within +/-15 %", "outcome / rejection reason"]
r = put(ws, 3, heads, bold=True, fill=GREY, wrap=True)
for _, t in TRIALS.iterrows():
    ws.cell(row=r, column=1, value=t["order_id"])
    ws.cell(row=r, column=2, value=int(t["trial_no"]))
    for c_ in (3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 17):
        ws.cell(row=r, column=c_).fill = YELLOW
    ws.cell(row=r, column=11, value=f'=IF(COUNT(H{r}:J{r})=3,MROUND(AVERAGE(H{r}:J{r}),0.5),"")')
    ws.cell(row=r, column=15, value=f'=IF(COUNT(L{r}:N{r})=3,MROUND(AVERAGE(L{r}:N{r}),0.5),"")')
    ws.cell(row=r, column=16, value=(f'=IF(COUNT(L{r}:N{r})=3,IF(AND(ABS(L{r}-AVERAGE(L{r}:N{r}))<=0.15*AVERAGE(L{r}:N{r}),'
                                     f'ABS(M{r}-AVERAGE(L{r}:N{r}))<=0.15*AVERAGE(L{r}:N{r}),'
                                     f'ABS(N{r}-AVERAGE(L{r}:N{r}))<=0.15*AVERAGE(L{r}:N{r})),"yes","NO - repeat (IS 516 cl 3.6)"),"")'))
    r += 1

# ------------------------------------------------------------------ Report checklist
ws = sheet("Report checklist", [110, 12])
r = put(ws, 1, ["Mix design report -- IS 10262 cl 5.8.1 (tick each)"], bold=True)
r = 3
for item in IS.IS10262_CL_5_8_1_REPORT:
    ws.cell(row=r, column=1, value=item).alignment = WRAP
    ws.cell(row=r, column=2).fill = YELLOW
    r += 1

# ------------------------------------------------------------------ Exploration
ws = sheet("Exploration", [18, 9, 9, 9, 9, 9, 9, 9, 9, 9, 50])
cols = ["order_id", "binder_kg_m3", "cement_kg_m3", "flyash_kg_m3", "water_kg_m3", "sp_pct_binder",
        "wc_counted", "fc_pred", "slump_pred", "slump_sd", "note"]
put(ws, 1, ["Optional exploration proposals -- where the models know least; expert review before casting"], bold=True)
df_block(ws, 3, EXPLORE, cols, ["order", "binder", "cement", "fly ash", "water", "SP %", "w/c counted",
                                "predicted 28 d", "predicted slump", "slump sd", "note"])

for w_ in wb.worksheets:
    w_.freeze_panes = None
wb.save("15_lab_trial_pack.xlsx")
print("wrote 15_lab_trial_pack.xlsx:", ", ".join(wb.sheetnames))

## 3. Check the workbook against the pipeline

The weighed water in every batch sheet is an Excel formula. Re-computing the same formula
in Python from the same inputs must give the value `layer0.batch_sheet` gave.

In [ ]:
bad = 0
for (oid, tno), g in BATCHES.groupby(["order_id", "trial_no"], sort=False):
    V = g["batch_m3"].iloc[0]
    a_s, a_c = g["sand_absorption_pct"].iloc[0], g["ca_absorption_pct"].iloc[0]
    m_s, m_c = g["sand_moisture_pct_used"].iloc[0], g["ca_moisture_pct_used"].iloc[0]
    w = g.loc[g.material.str.startswith("water"), "design_ssd_kg_m3"].iloc[0] * V
    for _, x in g.iterrows():
        ssd = x["design_ssd_kg_m3"] * V
        if x["material"].startswith("fine"):
            w += ssd - ssd / (1 + a_s / 100) * (1 + m_s / 100)
        elif x["material"].startswith("coarse"):
            w += ssd - ssd / (1 + a_c / 100) * (1 + m_c / 100)
    py = g.loc[g.material.str.startswith("water"), "batch_kg"].iloc[0]
    ok = abs(w - py) <= 0.02            # 0.01 kg rounding on each term in layer0
    bad += (not ok)
    print(f"{oid:18s} trial {int(tno)}: formula {w:7.3f} kg   layer0 {py:7.3f} kg   {'ok' if ok else 'MISMATCH'}")
assert bad == 0, "batch sheet formulas disagree with layer0.batch_sheet"
print("\nall batch-sheet formulas agree with Layer 0")